In [51]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict

In [52]:
load_dotenv()

True

In [53]:
model = ChatOllama(
    model = "gpt-oss:120b-cloud",
    base_url = "https://ollama.com",
    client_kwargs = {
        "headers":{
            "Authorization":f"Bearer {os.getenv('API_KEY')}"
        }
    }
)

In [54]:
class BlogState(TypedDict):
    topic:str
    outlines:str
    blog:str
    score:float

In [55]:
graph = StateGraph(BlogState)

In [56]:
def outline_llm(state:BlogState)->BlogState:
    topic = state['topic']
    prompt = f"Generate outlines for writing the Blog the topic of - {topic}"
    outlines = model.invoke(prompt).content
    state['outlines'] = outlines
    return state

In [57]:
def Blog_llm(state: BlogState)->BlogState:
    topic = state['topic']
    outlines = state['outlines']
    prompt = f"write the Blog on {topic} in the given detailed outlines {outlines}"
    blog = model.invoke(prompt).content
    state['blog'] = blog
    return state

In [58]:
def score_llm(state: BlogState)->BlogState:
    topic = state['topic']
    outlines = state['outlines']
    blog = state['blog']
    prompt = f"Based on the outline {outlines} for the topic {topic} acording to the the content {blog} score it(between 0 to 100 float no.)"
    score = model.invoke(prompt).content
    state['score'] = score
    return state

In [59]:
graph.add_node('outline_llm',outline_llm)
graph.add_node('Blog_llm',Blog_llm)
graph.add_node('Score_llm',score_llm)

graph.add_edge(START, 'outline_llm')
graph.add_edge('outline_llm','Blog_llm')
graph.add_edge('Blog_llm','Score_llm')
graph.add_edge('Score_llm',END)

workflow = graph.compile()

In [60]:
topi = "Favourite fast food"

answer = workflow.invoke({
    "topic": topi
})

print(answer["blog"])

**Meta Title:** My Favourite Fast Food – The Ultimate Guide to Chick‑Fil‑A’s Best Menu Items  
**Meta Description:** Discover why Chick‑Fil‑A is my go‑to fast‑food spot. From the iconic chicken sandwich to secret‑menu hacks, nutrition facts, and how it stacks up against the competition.  
**Tags:** fast food, favourite fast food, Chick‑Fil‑A review, secret menu, comfort food, fast‑food ranking, healthy fast food  

---

# Why I’m Obsessed with One Fast‑Food Joint (and You Might Be Too)

The first time I bit into a Chick‑Fil‑A chicken sandwich, it was a Tuesday after school. I’d saved every spare dime I could find, walked three blocks to the nearest drive‑thru, and watched the golden‑brown chicken glide onto the bun like a small, crispy miracle. The moment the warm bun, buttery spread, and perfectly seasoned chicken met my taste buds, a little voice inside me shouted, “This is it.”  

Since that day, the red‑and‑white logo has become my culinary compass. In this post you’ll learn:

* **

In [61]:
print(answer['outlines'])

Below are three ready‑to‑use **blog outlines** you can adapt to fit the tone, length, and audience you have in mind.  
Each outline is built around the central theme **“Favourite Fast Food”** but approaches it from a slightly different angle, so you can pick the one that feels most natural for you (or mix‑and‑match sections).

---

## 1️⃣ Outline #1 – *“My All‑Time Favourite Fast‑Food Spot: A Deep‑Dive”*  
*Great for a personal‑story, “hero‑brand” style post that also serves as a mini‑guide for readers.*

| Section | Headline (suggested) | Main Points / Sub‑bullets |
|--------|----------------------|---------------------------|
| **1. Hook / Intro** | “Why I’m Obsessed with One Fast‑Food Joint (and You Might Be Too)” | • Quick anecdote that captures the craving (e.g., first bite as a kid) <br>• Promise: what readers will learn (taste, secret menu, best order). |
| **2. The Fast‑Food Landscape** | “Fast Food 101: From Drive‑Thru to Gourmet‑Style Chains” | • Brief snapshot of the industr

In [62]:
print(answer['score'])

**Score:** 88.6 / 100  

*The post hits all key criteria: strong hook, clear structure, SEO‑focused meta tags, engaging sub‑heads, multimedia suggestions, nutritional data, secret‑menu tips, comparison chart, and a clear call‑to‑action. Minor improvements could be tighter keyword density and a few more internal/external links to boost authority.*
